# MCP Client (Streamable HTTP) – Tutorial

This notebook uses the **official MCP Python SDK** to talk to any **Streamable HTTP** MCP server. The same client works with:

| Server | Default URL |
|--------|-------------|
| **mcp-proxy** (`using-mcp-proxy/mcp-proxy/server.py`) | `http://localhost:8080/mcp` |
| **using-mcp-sdk** (`using-mcp-sdk/server_http.py`) | `http://localhost:8000/mcp` |

**Prerequisite:** Start one of the servers in another terminal, e.g.:

```bash
# Option A: MCP proxy (needs http-server backend for real GitHub calls)
cd using-mcp-proxy/mcp-proxy && python server.py

# Option B: SDK server (talks to GitHub directly)
cd using-mcp-sdk && python server_http.py
```

## 1. Dependencies, Imports and URL

Install what the MCP client needs.
We use `streamable_http_client` and `ClientSession` from the MCP SDK. Set `MCP_SERVER_URL` to the server you started (proxy = 8080, SDK = 8000).

In [9]:
!pip install -r requirements.txt


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [10]:
import json
import os

# Change to the server you have running:
# - mcp-proxy:    http://localhost:8080/mcp
# - SDK server:   http://localhost:8000/mcp
MCP_SERVER_URL = os.environ.get("MCP_SERVER_URL", "http://localhost:8000/mcp")

try:
    from mcp.client.session import ClientSession
    from mcp.client.streamable_http import streamable_http_client
except ImportError:
    raise SystemExit("Install the MCP SDK: pip install -r requirements.txt")

# One connection for the whole notebook (set in "Connect and initialize"; stays open until kernel restart)
_mcp_transport = None
_mcp_session = None

## 2. Connect and initialize (run once)

Open one Streamable HTTP connection and keep it for the rest of the notebook. Run this cell first; later cells reuse `_mcp_session`.

In [11]:
# Enter context once; keep transport and session for later cells (Jupyter's loop stays running).
_mcp_transport = streamable_http_client(MCP_SERVER_URL, terminate_on_close=False)
streams = await _mcp_transport.__aenter__()
# SDK may yield (read, write) or (read, write, get_session_id); we only need the first two.
_read_stream, _write_stream = streams[0], streams[1]
_mcp_session = ClientSession(_read_stream, _write_stream)
await _mcp_session.__aenter__()

init_result = await _mcp_session.initialize()
print("Initialize result:")
print(json.dumps(init_result.model_dump(), indent=2, default=str))

Initialize result:
{
  "meta": null,
  "protocolVersion": "2025-11-25",
  "capabilities": {
    "experimental": {},
    "logging": null,
    "prompts": {
      "listChanged": true
    },
    "resources": {
      "subscribe": false,
      "listChanged": true
    },
    "tools": {
      "listChanged": true
    },
    "completions": null,
    "tasks": null,
    "extensions": {
      "io.modelcontextprotocol/ui": {}
    }
  },
  "serverInfo": {
    "name": "github-pr-review",
    "title": null,
    "version": "3.0.2",
    "websiteUrl": null,
    "icons": null
  },
  "instructions": null
}


## 3. List tools (same session)

Uses the shared `_mcp_session` from the connect cell.

In [12]:
tools_result = await _mcp_session.list_tools()
print("Tools:")
for t in tools_result.tools:
    print(f"  - {t.name}: {t.description or '(no description)'}")

Tools:
  - get_pull_request_files: Get the list of files changed in a pull request with their diffs and change statistics.
  - create_pr_review: Create a review comment on a pull request. Use event COMMENT to add a comment, or APPROVE / REQUEST_CHANGES.


## 4. Call a tool: get_pull_request_files (same session)

Execute **get_pull_request_files** for a public repo (e.g. octocat/hello-world).

In [13]:
call_result = await _mcp_session.call_tool(
    "get_pull_request_files",
    arguments={"owner": "octocat", "repo": "hello-world", "pull_number": 1},
)
if getattr(call_result, "is_error", False):
    print("Tool error:", call_result.content)
else:
    for block in call_result.content:
        if hasattr(block, "text") and block.text:
            text = block.text
            print(text[:800] + ("..." if len(text) > 800 else ""))

[{'filename': 'README', 'status': 'modified', 'additions': 6, 'deletions': 1, 'changes': 7, 'patch': '@@ -1 +1,6 @@\n-Hello World!\n\\ No newline at end of file\n+Hello World!\n+$ mkdir ~/Hello-WorldCreates a directory for your project called "Hello-World" in your user directory\n+$ cd ~/Hello-WorldChanges the current working directory to your newly created directory\n+$ git initSets up the necessary Git files\n+Initialized empty Git repository in /Users/your_user_directory/Hello-World/.git/\n+$ touch README\n\\ No newline at end of file'}]


## 5. Call a tool: create_pr_review (same session)

**create_pr_review** posts a review comment. You need a repo with write access and `GITHUB_TOKEN` on the server. Uncomment and set owner/repo/pull_number to run it.

In [ ]:
# Uncomment and set your repo/PR to post a comment:
# result = await _mcp_session.call_tool(
#     "create_pr_review",
#     arguments={"owner": "myorg", "repo": "myrepo", "pull_number": 42, "event": "COMMENT", "body": "LGTM"},
# )
# print(result.content)
print("create_pr_review(owner, repo, pull_number, event, body) is available.")
print("Uncomment the lines above and set owner/repo/pull_number to run it.")

[TextContent(type='text', text="{'error': True, 'status_code': 404, 'message': {'message': 'Not Found', 'documentation_url': 'https://docs.github.com/rest/pulls/reviews#create-a-review-for-a-pull-request', 'status': '404'}}", annotations=None, meta=None)]
create_pr_review(owner, repo, pull_number, event, body) is available.
Uncomment the lines above and set owner/repo/pull_number to run it.


## 6. Connection lifetime

The transport and session use anyio cancel scopes that must be exited in the **same task** that entered them. We do **not** call `__aexit__` from a separate cell (that would raise "Attempted to exit cancel scope in a different task than it was entered in"). The connection stays open until you **restart the kernel** or close the notebook.

In [15]:
# Do not call __aexit__ here — it must run in the same task as __aenter__ (the connect cell).
# The connection is released when you restart the kernel.
print("Connection stays open until kernel restart.")

Connection stays open until kernel restart.


## Summary

- **Streamable HTTP** = one URL for POST (JSON-RPC) and GET (SSE). The MCP SDK’s `streamable_http_client(url)` handles both.
- Same client works for **mcp-proxy** (port 8080) and **using-mcp-sdk server_http** (port 8000); only the URL changes.
- This notebook reuses **one connection**: open in section 2, use `_mcp_session` in sections 3–5. The connection stays open until you restart the kernel (we don’t close it from another cell, because anyio requires the same task to exit the context).